In [1]:
!pip install -q huggingface_hub

In [2]:
from pathlib import Path

files = list(Path("/kaggle").rglob("download_visual_cot.py"))
files

[PosixPath('/kaggle/input/datasets/khoangoo/test-data/download_visual_cot.py')]

In [3]:
  !python /kaggle/input/datasets/khoangoo/test-data/download_visual_cot.py --mode light



=== README.md ===
  Dang tai ...
README.md: 4.11kB [00:00, 2.28MB/s]
  Da tai xong: visual-cot/README.md
  [i] File nay khong co hash LFS tren HF (co the la file thuong, vd README.md) -> bo qua buoc doi chieu hash, chi xac nhan file ton tai.

=== viscot_363k.json ===
  Dang tai ...
viscot_363k.json: 100%|██████████████████████| 345M/345M [00:03<00:00, 87.3MB/s]
  Da tai xong: visual-cot/viscot_363k.json
  Dang tinh SHA256 local (co the mat vai phut voi file lon)...
  [OK] Checksum khop.
       expected=39c2a901d357fcb1eb377f6046a9124ddb9ac97f1620ca73b87af0340a7f7f8b
       actual  =39c2a901d357fcb1eb377f6046a9124ddb9ac97f1620ca73b87af0340a7f7f8b

Hoan tat.
Du lieu nam trong: /kaggle/working/visual-cot
Luu y: mot so anh nguon (GQA, OCR-VQA, TextVQA...) trong bo Visual CoT yeu cau dang ky/tai rieng theo huong dan tai https://github.com/deepcs233/Visual-CoT#readme


In [4]:
import json
from pathlib import Path
from pprint import pprint
import ast
import json
from collections import Counter
from pathlib import Path

json_path = Path("/kaggle/working/visual-cot/viscot_363k.json")

with json_path.open("r", encoding="utf-8") as f:
    data = json.load(f)

print("Kiểu dữ liệu:", type(data))
print("Số phần tử:", len(data))

if isinstance(data, list):
    sample = data[0]
elif isinstance(data, dict):
    print("Các key cấp cao:", list(data.keys()))

    list_keys = [
        key for key, value in data.items()
        if isinstance(value, list) and len(value) > 0
    ]

    if list_keys:
        print("Danh sách nằm trong key:", list_keys)
        sample = data[list_keys[0]][0]
    else:
        sample = data
else:
    sample = data

print("\nMẫu dữ liệu đầu tiên:")
pprint(sample)

Kiểu dữ liệu: <class 'list'>
Số phần tử: 404120

Mẫu dữ liệu đầu tiên:
{'conversations': [{'from': 'human',
                    'value': '<image>\n'
                             'Can you tell me about the hairstyles of the '
                             'individuals in the image? Please provide the '
                             'bounding box coordinate of the region that can '
                             'help you answer the question better.'},
                   {'from': 'gpt', 'value': '[0.562, 0.228, 0.646, 0.292]'},
                   {'from': 'human', 'value': '<image>'},
                   {'from': 'gpt', 'value': 'They have shaggy hair.'}],
 'dataset': 'flickr30k',
 'image': ['cot/flickr30k/1000092795.jpg',
           'cot/flickr30k/1000092795.jpg###[198, 114, 240, 146]'],
 'question_id': 0,
 'split': 'train'}


In [5]:
with json_path.open("r", encoding="utf-8") as f:
    data = json.load(f)

dataset_counts = Counter()
split_counts = Counter()
conversation_length_counts = Counter()

valid_bbox_norm = 0
valid_crop_box = 0
invalid_samples = 0
missing_fields = Counter()

for sample in data:
    try:
        dataset_name = sample.get("dataset")
        split_name = sample.get("split")
        images = sample.get("image")
        conversations = sample.get("conversations")

        if dataset_name is None:
            missing_fields["dataset"] += 1
        else:
            dataset_counts[dataset_name] += 1

        if split_name is None:
            missing_fields["split"] += 1
        else:
            split_counts[split_name] += 1

        if not isinstance(images, list) or len(images) < 1:
            missing_fields["image"] += 1

        if not isinstance(conversations, list):
            missing_fields["conversations"] += 1
            invalid_samples += 1
            continue

        conversation_length_counts[len(conversations)] += 1

        # Kiểm tra normalized bbox trong câu trả lời GPT đầu tiên
        if len(conversations) >= 2:
            bbox_text = conversations[1].get("value", "")
            try:
                bbox = ast.literal_eval(bbox_text)
                if (
                    isinstance(bbox, list)
                    and len(bbox) == 4
                    and all(isinstance(v, (int, float)) for v in bbox)
                    and all(0 <= float(v) <= 1 for v in bbox)
                ):
                    valid_bbox_norm += 1
            except (ValueError, SyntaxError):
                pass

        # Kiểm tra pixel crop box trong image[1]
        if isinstance(images, list) and len(images) >= 2 and "###" in images[1]:
            _, crop_text = images[1].split("###", 1)
            try:
                crop_box = ast.literal_eval(crop_text)
                if (
                    isinstance(crop_box, list)
                    and len(crop_box) == 4
                    and all(isinstance(v, (int, float)) for v in crop_box)
                ):
                    valid_crop_box += 1
            except (ValueError, SyntaxError):
                pass

    except Exception:
        invalid_samples += 1

print("TỔNG SỐ MẪU:", len(data))

print("\nPHÂN BỐ SPLIT:")
for name, count in split_counts.most_common():
    print(f"{name:20s}: {count:,}")

print("\nTOP DATASET NGUỒN:")
for name, count in dataset_counts.most_common(30):
    print(f"{name:30s}: {count:,}")

print("\nĐỘ DÀI CONVERSATION:")
for length, count in sorted(conversation_length_counts.items()):
    print(f"{length} messages: {count:,}")

print("\nKIỂM TRA ANNOTATION:")
print(f"Normalized bbox hợp lệ : {valid_bbox_norm:,}")
print(f"Pixel crop box hợp lệ   : {valid_crop_box:,}")
print(f"Mẫu lỗi                 : {invalid_samples:,}")
print(f"Thiếu trường             : {dict(missing_fields)}")

TỔNG SỐ MẪU: 404120

PHÂN BỐ SPLIT:
train               : 404,120

TOP DATASET NGUỒN:
flickr30k                     : 135,735
gqa                           : 88,294
openimages                    : 43,053
docvqa                        : 33,453
textcap                       : 32,152
v7w                           : 30,491
textvqa                       : 18,524
infographicsvqa               : 15,055
cub                           : 3,987
vsr                           : 3,376

ĐỘ DÀI CONVERSATION:
4 messages: 404,120

KIỂM TRA ANNOTATION:
Normalized bbox hợp lệ : 403,249
Pixel crop box hợp lệ   : 404,120
Mẫu lỗi                 : 0
Thiếu trường             : {}


In [6]:
import pandas as pd

source_summary = pd.DataFrame(
    dataset_counts.most_common(),
    columns=["dataset", "num_samples"]
)

source_summary["percentage"] = (
    source_summary["num_samples"] / len(data) * 100
).round(3)

source_summary.to_csv(
    "/kaggle/working/visual-cot/source_summary.csv",
    index=False
)

display(source_summary)

,dataset,num_samples,percentage
0,flickr30k,135735,33.588
1,gqa,88294,21.848
2,openimages,43053,10.654
3,docvqa,33453,8.278
4,textcap,32152,7.956
5,v7w,30491,7.545
6,textvqa,18524,4.584
7,infographicsvqa,15055,3.725
8,cub,3987,0.987
9,vsr,3376,0.835


In [7]:
from huggingface_hub import hf_hub_download
from pathlib import Path

REPO_ID = "deepcs233/Visual-CoT"
OUT_DIR = Path("/kaggle/working/visual-cot")

files = [
    "cot_with_detailed_reasoning_steps/gqa_cot_train.jsonl",
    "cot_with_detailed_reasoning_steps/gqa_cot_val.jsonl",
]

for filename in files:
    path = hf_hub_download(
        repo_id=REPO_ID,
        repo_type="dataset",
        filename=filename,
        local_dir=str(OUT_DIR),
    )
    print("Đã tải:", path)

cot_with_detailed_reasoning_steps/gqa_co(…):   0%|          | 0.00/71.0M [00:00<?, ?B/s]

Đã tải: /kaggle/working/visual-cot/cot_with_detailed_reasoning_steps/gqa_cot_train.jsonl


gqa_cot_val.jsonl: 0.00B [00:00, ?B/s]

Đã tải: /kaggle/working/visual-cot/cot_with_detailed_reasoning_steps/gqa_cot_val.jsonl


In [8]:
import json
from pathlib import Path
from pprint import pprint

train_path = Path(
    "/kaggle/working/visual-cot/"
    "cot_with_detailed_reasoning_steps/gqa_cot_train.jsonl"
)

records = []

with train_path.open("r", encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if line:
            records.append(json.loads(line))

print("Số mẫu train:", len(records))
pprint(records[0])

Số mẫu train: 88294
{'answer': 'girl',
 'bboxs': [[214, 0, 433, 374]],
 'dataset': 'gqa',
 'full_answer': 'The girl is wearing a shirt.',
 'height': 375,
 'image': '2331819.jpg',
 'question': 'Who is wearing a shirt?',
 'reasoning': [{'argument': 'shirt (4653737)',
                'dependencies': [],
                'operation': 'select'},
               {'argument': 'person,wearing,s (4653736)',
                'dependencies': [0],
                'operation': 'relate'},
               {'argument': 'name', 'dependencies': [1], 'operation': 'query'}],
 'split': 'train',
 'thought': '1. We are looking for someone who is wearing a shirt in the '
            "image. 2. First, let's identify all the people in the image. 3. "
            'Next, we need to determine who among the people is wearing a '
            'shirt. 4. After identifying the person, we can state the answer '
            'based on the observation from the image.',
 'width': 500}


In [9]:
def extract_image_path(sample):
    image = sample.get("image")

    if isinstance(image, str):
        return image.split("###", 1)[0]

    if isinstance(image, list) and image:
        return image[0].split("###", 1)[0]

    return None


image_paths = {
    extract_image_path(sample)
    for sample in records[:5000]
}

image_paths.discard(None)

print("Số sample kiểm tra:", min(5000, len(records)))
print("Số ảnh duy nhất:", len(image_paths))
print("Một số đường dẫn:")
for path in list(sorted(image_paths))[:10]:
    print(path)

Số sample kiểm tra: 5000
Số ảnh duy nhất: 3856
Một số đường dẫn:
10.jpg
1029.jpg
1045.jpg
1051.jpg
107910.jpg
107916.jpg
107929.jpg
107941.jpg
107990.jpg
1159.jpg
